# Alerts — Detection-Only Historical Backfill

Runs **Stage 1 (detect) only** for every date in a range and writes the raw signals
to a Snowflake table.  No facts, narratives, or scoring.

| Cell | What happens |
|------|--------------|
| 1 | Config — date range, output table, optional store filter |
| 2 | Session — connect to Snowflake |
| 3 | Discover — dates to process vs already written |
| 4 | Run — detect + write signals per date |

Output table schema: `RUN_DATE, SIGNAL_TYPE, STORE_ID, DEPARTMENT, TRIGGER_TYPES,`
`THIS_WEEK_LSV, LAST_WEEK_OSA, THIS_WEEK_OSA, WOW_DRIFT_PP, CHAIN_THIS_WEEK_OSA,`
`CHAIN_GAP_PP, CHAIN_WOW_DRIFT_PP, REPEAT_ITEM_COUNT,`
`THIS_WEEK_TF_RATE, LAST_WEEK_TF_RATE, TF_RATE_CHANGE, CHAIN_TF_RATE, ABOVE_CHAIN_BY`

In [ ]:
# ── Date range ─────────────────────────────────────────────────────────────────
START_DATE_STR = '2026-03-01'
END_DATE_STR   = '2026-03-28'

# ── Client config (None = base defaults; 'eastofengland' = client override) ────
CLIENT = None   # or 'eastofengland'

# ── Optional: restrict to one store for a fast smoke test ─────────────────────
QC_STORE_ID = None   # e.g. 507 — set to None for all stores

# ── Output table override (None = use config; string = explicit table) ────────
OUTPUT_TABLE_OVERRIDE = None   # e.g. 'SNOWFLAKE_LEARNING_DB.STORE_INSIGHT_2.ALERTS_SIGNALS'

import datetime

start_date = datetime.date.fromisoformat(START_DATE_STR)
end_date   = datetime.date.fromisoformat(END_DATE_STR)

# Deferred: table names resolved after config loaded in next cell
print(f'Date range   : {start_date} – {end_date}  ({(end_date - start_date).days + 1} days)')
print(f'Client       : {CLIENT or "base defaults"}')
print(f'Store filter : {QC_STORE_ID if QC_STORE_ID else "all stores"}')

Date range   : 2026-03-01 – 2026-03-28  (28 days)
Output table : SNOWFLAKE_LEARNING_DB.STORE_INSIGHT_2.ALERTS_SIGNALS
Store filter : all stores


In [ ]:
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

# ── Suppress noisy Snowflake connector output ─────────────────────────────────
logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.alerts.config.loader import load_config
from store_insight.alerts.pipeline.session import create_session

cfg = load_config(CLIENT)
print(f'Config loaded: client={cfg["client_name"]}')

# ── Resolve table names from config ──────────────────────────────────────────
OSA_TABLE       = cfg['tables']['osa']
HIERARCHY_TABLE = cfg['tables']['hierarchy']
FEEDBACK_TABLE  = cfg['tables']['feedback']
OUTPUT_TABLE    = OUTPUT_TABLE_OVERRIDE or cfg['output']['signals']

print(f'OSA table      : {OSA_TABLE}')
print(f'Hierarchy table: {HIERARCHY_TABLE}')
print(f'Feedback table : {FEEDBACK_TABLE}')
print(f'Output table   : {OUTPUT_TABLE}')

session = create_session(cfg)   # prints 'Snowflake session created.'

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/93a31d5d-1a5d-433f-baa0-138549cd2108/saml2?SAMLRequest=nZJBb%2BIwEIX%2FSuQ9x7GTUIEFVCwUNWp3F0EKUm9uYsAlsVOPQ0p%2F%2FToBpO6hPewliuw38z3Pm%2BHte1l4R2FAajVCFBPkCZXpXKrdCD2lc7%2BPPLBc5bzQSozQSQC6HQ%2BBl0XFJrXdq6V4qwVYzzVSwNqLEaqNYpqDBKZ4KYDZjK0mvx5ZiAnjAMJYh0OXkhykY%2B2trVgQNE2DmwhrswtCQkhABoFTtZIf6BOi%2Bp5RGW11potrybt70xcIGpC4RTiFIywuhT%2BlOo%2FgO8rLWQTsPk0X%2FuLPKkXe5Pq6qVZQl8KshDnKTDwtH88GwDmAkt5EJMb1wQftZoj5R20EBqWbbcEPItNlVVvXGru%2FYCvyoNA76QaWzEaoOsg8pXcbPn%2Bdlm9SbfRD9pE8l3bJpw9yfbw7ntZ6v6nEZr2BpMmQt77GG7bxJgC1SFQbqnVHJLzxSeyTMKWE9SgjPUzi6Bl5MxeqVNx2lVfnnQ9cysxo0FurVSGV6FwOIh7RvJf7lLtPHEVb%2F4Vz4tOo34sHWR5S0g%2Fa6EJ0Xh%2FWGTHj%2FxrKMPjc4rKOv11CyWyhC5mdvLk2JbdfB0gx7U5k7m87KRMll8Ukz40AcEEWhW6mRnDrtt6aWqBgfK

 pip install snowflake-connector-python[secure-local-storage]


In [17]:
# ── Find dates already written ─────────────────────────────────────────────────
def _get_existing_run_dates(session, table):
    try:
        rows = session.sql(f'SELECT DISTINCT RUN_DATE FROM {table}').collect()
        return {r[0] for r in rows}
    except Exception:
        return set()

existing = _get_existing_run_dates(session, OUTPUT_TABLE)

all_dates = []
d = start_date
while d <= end_date:
    all_dates.append(d)
    d += datetime.timedelta(days=1)

skipped = [d for d in all_dates if str(d) in existing]
pending = [d for d in all_dates if str(d) not in existing]

print(f'Total dates in range : {len(all_dates)}')
print(f'Already written      : {len(skipped)}')
print(f'To process           : {len(pending)}')
if pending:
    print(f'  First pending      : {pending[0]}')
    print(f'  Last pending       : {pending[-1]}')

Total dates in range : 28
Already written      : 1
To process           : 27
  First pending      : 2026-03-02
  Last pending       : 2026-03-28


In [ ]:
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import (
    StructType, StructField, StringType, IntegerType, FloatType,
)
from store_insight.alerts.pipeline.run_batch   import load_osa, load_feedback
from store_insight.alerts.stages._1_detect.detect import detect_signals

SIGNALS_SCHEMA = StructType([
    StructField('RUN_DATE',             StringType()),
    StructField('SIGNAL_TYPE',          StringType()),
    StructField('STORE_ID',             IntegerType()),
    StructField('DEPARTMENT',           StringType()),
    StructField('TRIGGER_TYPES',        StringType()),
    # OSA fields
    StructField('THIS_WEEK_LSV',        FloatType()),
    StructField('LAST_WEEK_OSA',        FloatType()),
    StructField('THIS_WEEK_OSA',        FloatType()),
    StructField('WOW_DRIFT_PP',         FloatType()),
    StructField('CHAIN_THIS_WEEK_OSA',  FloatType()),
    StructField('CHAIN_GAP_PP',         FloatType()),
    StructField('CHAIN_WOW_DRIFT_PP',   FloatType()),
    StructField('REPEAT_ITEM_COUNT',    IntegerType()),
    # Compliance fields
    StructField('THIS_WEEK_TF_RATE',    FloatType()),
    StructField('LAST_WEEK_TF_RATE',    FloatType()),
    StructField('TF_RATE_CHANGE',       FloatType()),
    StructField('CHAIN_TF_RATE',        FloatType()),
    StructField('ABOVE_CHAIN_BY',       FloatType()),
])


def _signals_to_rows(signals, run_date):
    rows = []
    for s in signals:
        if s['signal_type'] == 'OSA':
            rows.append((
                run_date,
                'OSA',
                s['store_id'],
                s.get('department'),
                ','.join(sorted(s['trigger_types'])),
                float(s['this_week_lsv'])       if s.get('this_week_lsv')       is not None else None,
                float(s['last_week_osa'])        if s.get('last_week_osa')        is not None else None,
                float(s['this_week_osa'])        if s.get('this_week_osa')        is not None else None,
                float(s['wow_drift_pp'])         if s.get('wow_drift_pp')         is not None else None,
                float(s['chain_this_week_osa'])  if s.get('chain_this_week_osa')  is not None else None,
                float(s['chain_gap_pp'])         if s.get('chain_gap_pp')         is not None else None,
                float(s['chain_wow_drift_pp'])   if s.get('chain_wow_drift_pp')   is not None else None,
                int(s['repeat_item_count'])      if s.get('repeat_item_count')    is not None else None,
                None, None, None, None, None,  # compliance fields
            ))
        else:  # COMPLIANCE
            rows.append((
                run_date,
                'COMPLIANCE',
                s['store_id'],
                None,
                ','.join(sorted(s['trigger_types'])),
                None, None, None, None, None, None, None, None,  # OSA fields
                float(s['this_week_tf_rate']),
                float(s['last_week_tf_rate']),
                float(s['tf_rate_change']),
                float(s['chain_tf_rate']),
                float(s['above_chain_by']),
            ))
    return rows


def _write_signals(session, rows, table):
    if not rows:
        print('  No signals to write.')
        return
    df = session.create_dataframe(rows, schema=SIGNALS_SCHEMA)
    df.write.mode('append').save_as_table(table)
    print(f'  Written {len(rows)} signal rows → {table}')


if not pending:
    print('Nothing to process — all dates already written.')
else:
    errors = []

    for i, analysis_date in enumerate(pending, 1):
        this_week_end   = str(analysis_date)
        this_week_start = str(analysis_date - datetime.timedelta(days=6))
        last_week_end   = str(analysis_date - datetime.timedelta(days=7))
        last_week_start = str(analysis_date - datetime.timedelta(days=13))
        lookback_start  = str(analysis_date - datetime.timedelta(days=69))

        print(f'\n── Date {i}/{len(pending)}: {analysis_date} ──')

        try:
            osa      = load_osa(session, lookback_start, this_week_end, cfg=cfg)
            feedback = load_feedback(session, last_week_start, this_week_end, cfg=cfg)

            signals = detect_signals(
                osa,
                last_week_start=last_week_start,
                last_week_end=last_week_end,
                this_week_start=this_week_start,
                this_week_end=this_week_end,
                lookback_start=lookback_start,
                store_id=QC_STORE_ID,
                feedback=feedback,
                cfg=cfg,
            )

            rows = _signals_to_rows(signals, run_date=this_week_end)
            _write_signals(session, rows, OUTPUT_TABLE)

        except Exception as exc:
            print(f'  ERROR on {analysis_date}: {exc}')
            errors.append((analysis_date, exc))

    print(f'\n══ Done ══')
    print(f'Processed : {len(pending) - len(errors)}')
    print(f'Errors    : {len(errors)}')
    if errors:
        for d, e in errors:
            print(f'  {d} → {e}')


── Date 1/27: 2026-03-02 ──
[1_detect] TICK_AND_FLICK=13 (chain avg 33.6%)
[1_detect] 342 signals — REPEAT=129, VS_CHAIN=140, WOW=201, TICK_AND_FLICK=13
  Written 342 signal rows → SNOWFLAKE_LEARNING_DB.STORE_INSIGHT_2.ALERTS_SIGNALS

── Date 2/27: 2026-03-03 ──
[1_detect] TICK_AND_FLICK=15 (chain avg 33.3%)
[1_detect] 347 signals — REPEAT=130, VS_CHAIN=145, WOW=202, TICK_AND_FLICK=15
  Written 347 signal rows → SNOWFLAKE_LEARNING_DB.STORE_INSIGHT_2.ALERTS_SIGNALS

── Date 3/27: 2026-03-04 ──
[1_detect] TICK_AND_FLICK=17 (chain avg 33.3%)
[1_detect] 350 signals — REPEAT=128, VS_CHAIN=145, WOW=207, TICK_AND_FLICK=17
  Written 350 signal rows → SNOWFLAKE_LEARNING_DB.STORE_INSIGHT_2.ALERTS_SIGNALS

── Date 4/27: 2026-03-05 ──
[1_detect] TICK_AND_FLICK=23 (chain avg 33.6%)
[1_detect] 369 signals — REPEAT=130, VS_CHAIN=144, WOW=222, TICK_AND_FLICK=23
  Written 369 signal rows → SNOWFLAKE_LEARNING_DB.STORE_INSIGHT_2.ALERTS_SIGNALS

── Date 5/27: 2026-03-06 ──
[1_detect] TICK_AND_FLICK=22 (c